# Predicting late deliveries
Protocol: `protocol.yaml`. Temporal holdout from `holdout.csv`, walk-forward CV (5 splits) on the training rows, model chosen by mean CV ROC AUC.

Excluded columns: `consignment` (identifier), `dispatched_on` (split only), and `claim_paid_eur` - a claim is settled after delivery, so it is not known at prediction time and leaks the outcome.

In [ ]:
import json, hashlib, numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='859a474044a292565d72b06a6180ade1c1adbc8c72a47c4a518186876f581f27'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='fae5c02170970fe92fcadaee6d1109bb1d459c9dc033dfc487bb2c823d0f6736'
df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==8051 and len(hold_idx)==1610
train = df.drop(index=hold_idx)
train = train.assign(_t=pd.to_datetime(train.dispatched_on)).sort_values('_t', kind='stable')
hold = df.loc[np.sort(hold_idx)]
print(len(train), len(hold), train._t.max(), hold.dispatched_on.min())

In [ ]:
CAT = ['carrier']; NUM = ['distance_km','weight_kg','parcels']
FEATURES = CAT + NUM
y_tr = train.late_delivery.to_numpy(); X_tr = train[FEATURES]

def pre(scale):
    return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), CAT),
                              ('n', StandardScaler() if scale else 'passthrough', NUM)])

# all candidates are named before any is scored
candidates = {
    'baseline_prior': Pipeline([('p', pre(False)), ('m', DummyClassifier(strategy='prior'))]),
    'logreg_C1': Pipeline([('p', pre(True)), ('m', LogisticRegression(C=1.0, max_iter=1000))]),
    'logreg_C0.1': Pipeline([('p', pre(True)), ('m', LogisticRegression(C=0.1, max_iter=1000))]),
    'random_forest': Pipeline([('p', pre(False)), ('m', RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1))]),
    'hist_gb': Pipeline([('p', pre(False)), ('m', HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0))]),
}
print(list(candidates))

In [ ]:
folds = list(TimeSeriesSplit(n_splits=5).split(X_tr))
orig = train.index.to_numpy()  # 0-based row positions in data.csv
cv_scores = {}
for name, m in candidates.items():
    s = []
    for tr, va in folds:
        mm = clone(m).fit(X_tr.iloc[tr], y_tr[tr])   # preprocessing is fitted inside the fold
        s.append(float(roc_auc_score(y_tr[va], mm.predict_proba(X_tr.iloc[va])[:,1])))
    cv_scores[name] = s
print(pd.DataFrame({k: dict(mean=np.mean(v), std=np.std(v)) for k, v in cv_scores.items()}).T)
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print('chosen', chosen)

In [ ]:
final = clone(candidates[chosen]).fit(X_tr, y_tr)
holdout_auc = float(roc_auc_score(hold.late_delivery, final.predict_proba(hold[FEATURES])[:,1]))
print('holdout roc_auc', holdout_auc)

In [ ]:
evidence = dict(
    features=FEATURES,
    candidates=list(candidates),
    cv_folds=[[[int(i) for i in orig[tr]], [int(i) for i in orig[va]]] for tr, va in folds],
    cv_scores=cv_scores,
    chosen=chosen,
    holdout_rows=[int(i) for i in hold.index],
    holdout_scores={chosen: holdout_auc},
)
json.dump(evidence, open('evidence.json','w'))

## Conclusion
See the CV table and the holdout ROC AUC above (0.5 = no skill).